# 🚀 Dual Model Trainer (Segmentor + Classifier)

In this notebook, we implement an end-to-end Dual Model training and evaluation system:
1. **Stage 1 (Segmentor):** A YOLOv12m-seg (or any YOLO architecture) instance segmentation model trained on a generalized "1-class Rock" dataset.
2. **Stage 2 (Classifier):** Fine-grained Image Classification models trained natively on rock instance *crops*.

We benchmark 3 Classifier architectures specifically fine-tuned for small rock crops:
- `YOLOv26-cls`
- `EfficientNet-B0` (PyTorch `timm` / `torchvision`)
- `ViT-B/16` (Vision Transformer, `torchvision`)

**Note:** The dataset generation steps (converting generalized 6-class YOLO dataset into single class &amp; cropping bounding boxes into class folders) are assumed to be complete. 

In [ ]:
from pathlib import Path
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from torchvision.models import efficientnet_b0, vit_b_16
import copy
from ultralytics import YOLO

# ==========================================
# 📝 PATH CONFIGURATION - Adjust as necessary
# ==========================================
BASE_DIR = Path().resolve().parent.parent

# ── 1. SEGMENTOR PATHS ──
# YOLO dataset containing only 1-class mapping ("rock") 
SEGMENTOR_DATASET_YAML = BASE_DIR / "datasets" / "batch4_single_class" / "data.yaml"

# ── 2. CLASSIFIER PATHS ──
# Structured as PyTorch ImageFolder (e.g. train/Silt/..., val/Sandstone/...)
CLASSIFIER_DATA_ROOT = BASE_DIR / "datasets" / "classifier_crops"

# ── TARGET MODEL NAMES ──
SEGMENTOR_RUN_NAME = "YOLO_Segmentor_SingleClass_Final"
MODELS_DIR = BASE_DIR / "models"
MODELS_DIR.mkdir(parents=True, exist_ok=True)

# Basic Setup Validation
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device   : {device}")
print(f"Base Dir : {BASE_DIR}")
print(f"CUDA     : {torch.cuda.is_available()}")

## 🔧 Stage 1: Segmentor Training (Single-Class YOLO)
> **Goal**: Train YOLO to detect general boundaries of "Rocks", completely ignoring fine-grained taxonomy.

In [ ]:
def train_segmentor(epochs=100, imgsz=960, batch=16):
    """
    Trains the Model 1 (Segmentor) relying on Ultralytics YOLO logic natively.
    """
    print("\n" + "=" * 50)
    print("🚀 TRAINING STAGE 1: SEGMENTOR")
    print("=" * 50)
    
    # Optional check if dataset YAML exists before attempting
    if not SEGMENTOR_DATASET_YAML.exists():
        print(f"⚠️ Warning: Cannot initiate Stage 1 - Dataset not found: {SEGMENTOR_DATASET_YAML}")
        return None
        
    model = YOLO("yolo11m-seg.pt") # fallback/starter
    
    results = model.train(
        data=str(SEGMENTOR_DATASET_YAML),
        project=str(MODELS_DIR),
        name=SEGMENTOR_RUN_NAME,
        epochs=epochs,
        imgsz=imgsz,
        batch=batch,
        device=0 if device == 'cuda' else 'cpu',
        single_cls=True,  # Critical: Forces YOLO to treat all existing ground-truths as class 0
        exist_ok=True,
        verbose=False
    )
    return results

seg_results = train_segmentor(epochs=50)

## 🧠 Stage 2: Classifier Training Data Loader
> **Goal**: Preload and transform our cropped rock patches for PyTorch models

In [ ]:
# PyTorch ImageFolder structure requires Train &amp; Val loaders
batch_size = 32

data_transforms = {
    'train': transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.RandomHorizontalFlip(),
        transforms.RandomRotation(20),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ]),
    'val': transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ]),
}

def load_classifier_datasets():
    if not CLASSIFIER_DATA_ROOT.exists():
        print(f"⚠️ Warning: Crop directory {CLASSIFIER_DATA_ROOT} missing.")
        return None, None, None

    image_datasets = {
        x: datasets.ImageFolder(os.path.join(CLASSIFIER_DATA_ROOT, x), data_transforms[x]) 
        for x in ['train', 'val']
    }
    
    dataloaders = {
        x: DataLoader(image_datasets[x], batch_size=batch_size, shuffle=True, num_workers=4) 
        for x in ['train', 'val']
    }
    
    dataset_sizes = {x: len(image_datasets[x]) for x in ['train', 'val']}
    class_names = image_datasets['train'].classes
    return dataloaders, dataset_sizes, class_names

dataloaders, dataset_sizes, class_names = load_classifier_datasets()

## 🚀 Stage 2: Train Expert Classifiers

Here we build a standard reusable PyTorch loop and use Ultralytics for the YOLO-cls version. 

### A) PyTorch Reusable Training Loop (EfficientNet &amp; ViT)

In [ ]:
def train_pytorch_model(model, dataloaders, dataset_sizes, criterion, optimizer, num_epochs=15):
    """
    Standard PyTorch training module for ImageFolder datasets.
    Calculates Accuracy, runs evaluation step per epoch.
    """
    best_model_wts = copy.deepcopy(model.state_dict())
    best_acc = 0.0

    for epoch in range(num_epochs):
        print(f'Epoch {epoch}/{num_epochs - 1}')
        print('-' * 10)

        for phase in ['train', 'val']:
            if phase == 'train':
                model.train()
            else:
                model.eval()

            running_loss = 0.0
            running_corrects = 0

            for inputs, labels in dataloaders[phase]:
                inputs, labels = inputs.to(device), labels.to(device)
                optimizer.zero_grad()

                with torch.set_grad_enabled(phase == 'train'):
                    outputs = model(inputs)
                    _, preds = torch.max(outputs, 1)
                    loss = criterion(outputs, labels)

                    if phase == 'train':
                        loss.backward()
                        optimizer.step()

                running_loss += loss.item() * inputs.size(0)
                running_corrects += torch.sum(preds == labels.data)

            epoch_loss = running_loss / dataset_sizes[phase]
            epoch_acc = running_corrects.double() / dataset_sizes[phase]

            print(f'{phase} Loss: {epoch_loss:.4f} Acc: {epoch_acc:.4f}')

            if phase == 'val' and epoch_acc > best_acc:
                best_acc = epoch_acc
                best_model_wts = copy.deepcopy(model.state_dict())

    print(f'✅ Best val Acc: {best_acc:4f}')
    model.load_state_dict(best_model_wts)
    return model

### B) Classifier 1: Ultralytics YOLO-cls

In [ ]:
def train_yolo_classifier(epochs=50):
    print("\n" + "=" * 50)
    print("🚀 TRAINING STAGE 2: YOLO-cls")
    print("=" * 50)
    
    if not CLASSIFIER_DATA_ROOT.exists(): return None
        
    model = YOLO('yolo26n-cls.pt')
    results = model.train(
        data=str(CLASSIFIER_DATA_ROOT),
        epochs=epochs,
        imgsz=224,
        project=str(MODELS_DIR),
        name="Expert_YOLOcls_Final"
    )
    return results

cls_yolo_res = train_yolo_classifier(epochs=50)

### C) Classifier 2 &amp; 3: EfficientNet-B0 and ViT-B (Vision Transformer)

In [ ]:
def train_efficientnet(dataloaders, dataset_sizes, class_names, num_epochs=20):
    print("\n" + "=" * 50)
    print("🚀 TRAINING STAGE 2: EfficientNet-B0")
    print("=" * 50)
    
    model = efficientnet_b0(weights='IMAGENET1K_V1')
    
    # Replace final head mapping for num_classes
    num_ftrs = model.classifier[1].in_features
    model.classifier[1] = nn.Linear(num_ftrs, len(class_names))
    model = model.to(device)

    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=0.001)

    model = train_pytorch_model(model, dataloaders, dataset_sizes, criterion, optimizer, num_epochs=num_epochs)
    
    # Save the PyTorch weights
    torch.save(model, str(MODELS_DIR / "Expert_EfficientNet_Final.pt"))
    return model

def train_vit(dataloaders, dataset_sizes, class_names, num_epochs=20):
    print("\n" + "=" * 50)
    print("🚀 TRAINING STAGE 2: ViT-B/16")
    print("=" * 50)
    
    model = vit_b_16(weights='IMAGENET1K_V1')
    
    # Replace final head mapping for num_classes
    num_ftrs = model.heads.head.in_features
    model.heads.head = nn.Linear(num_ftrs, len(class_names))
    model = model.to(device)

    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=0.0001)  # Lower LR for ViT tuning

    model = train_pytorch_model(model, dataloaders, dataset_sizes, criterion, optimizer, num_epochs=num_epochs)
    
    # Save the PyTorch weights
    torch.save(model, str(MODELS_DIR / "Expert_ViT_Final.pt"))
    return model

efficientnet_model = train_efficientnet(dataloaders, dataset_sizes, class_names)
vit_model = train_vit(dataloaders, dataset_sizes, class_names)

## 📊 End-to-End Test Evaluation
We evaluate the complete system (Segmentor + Classifier) metrics: **Precision, Recall, F1-Score, and mAP**.

Since doing this directly requires matching predicted segment masks tightly with test set labels using IoU logic on our custom classes, below is a mock script detailing how `sklearn.metrics` will calculate this out of our Integration pipeline predictions.

In [ ]:
from sklearn.metrics import classification_report, precision_score, recall_score, f1_score

def mock_evaluation_report(y_true, y_pred, class_names):
    """
    Computes strict metrics for validation.
    y_true: list of integer target labels for crops evaluated
    y_pred: list of integer predictions mapped by the classifier
    """
    p = precision_score(y_true, y_pred, average='weighted')
    r = recall_score(y_true, y_pred, average='weighted')
    f1 = f1_score(y_true, y_pred, average='weighted')
    
    print("="*40)
    print("      INTEGRATION APP EVALUATION      ")
    print("="*40)
    print(f"mAP (Approximated):  ~{f1*100:.2f}%")
    print(f"Precision:           {p*100:.2f}%")
    print(f"Recall:              {r*100:.2f}%")
    print(f"F1-Score:            {f1*100:.2f}%")
    print("="*40)
    print("\nDetailed Classification Report:")
    print(classification_report(y_true, y_pred, target_names=class_names))


## 🎯 Integration Pipeline &amp; Visualizations

In [ ]:
import sys
sys.path.append(str(BASE_DIR))
from src.inference_dual_model import DualModelPipeline, DualModelVisualizer

# Define paths (ensure these paths exist after running training above)
SEG_MODEL_PATH = str(MODELS_DIR / SEGMENTOR_RUN_NAME / "weights" / "best.pt")
YOLO_CLS_PATH  = str(MODELS_DIR / "Expert_YOLOcls_Final" / "weights" / "best.pt")
EFFNET_PATH    = str(MODELS_DIR / "Expert_EfficientNet_Final.pt")

# Master Schema Map (7 Classes)
class_map = {
    0: 'CLAS - Silt',
    1: 'CLAS - Sandstone', 
    2: 'CARB - Limestone',
    3: 'ORG - Coal',
    4: 'CLAS - Shalestone',
    5: 'MIN - Quartz',
    6: 'ART - Cement'
}

configs = [
    {'name': 'YOLO-cls', 'model': YOLO_CLS_PATH, 'type': 'yolo', 'class_names': class_map},
    {'name': 'EfficientNet', 'model': EFFNET_PATH, 'type': 'pytorch', 'class_names': class_map}
]

def run_integration_visualization(image_path, target_classifier='YOLO-cls'):
    # Check dependencies locally before executing
    if not os.path.exists(SEG_MODEL_PATH) or not os.path.exists(YOLO_CLS_PATH):
        print(f"⚠️ Models not found. Train the models above first before initializing inference.")
        return
        
    # Init Reusable Class 
    pipeline = DualModelPipeline(segmentor_path=SEG_MODEL_PATH, classifier_configs=configs)
    
    # 1. Run Pipeline
    results, predictions = pipeline.predict(image=image_path, classifier_name=target_classifier)
    
    # 2. Draw Post-Processed Visualization
    visualizer = DualModelVisualizer(color_map=None)
    import cv2
    img = cv2.imread(image_path)
    drawn_image = visualizer.draw(image=img, predictions=predictions, thickness=2)
    
    # 3. Output
    from IPython.display import display, Image
    out_path = f"dual_model_{target_classifier}_pred.jpg"
    cv2.imwrite(out_path, drawn_image)
    print(f"✅ Segmented &amp; Classified Prediction exported as '{out_path}'")
    display(Image(filename=out_path))

# Example usage on Test Set
run_integration_visualization(str(BASE_DIR / "datasets" / "batch4" / "images" / "test_001.jpg"), target_classifier='EfficientNet')